# Stage graphs: the core vLLM-Omni abstraction

## Learning goals
- Read a pipeline as nodes (stages) and edges (transfer functions)
- Build and run a custom stage graph
- See where transfer functions transform inter-stage data

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## Nodes and edges

vLLM-Omni models any pipeline as a graph: **nodes** are stages (an AR LLM, a DiT), **edges** carry data between them through a connector, optionally applying a **transfer function** that reshapes one stage's output into the next stage's input. The voice pipeline is the linear case `Thinker -> Talker -> Code2Wav`, but the abstraction is general.

In [ ]:
from omni_tutorial import Stage, StageGraph, Connector
# Build a custom 2-stage graph with an explicit transfer function on the edge.
encode = Stage('encode', 'AR', lambda text: {'tokens': text.split()})
def to_summary(payload):
    return {'summary': ' '.join(payload['tokens'][:3]) + ' ...'}
summarize = Stage('summarize', 'AR', lambda p: {'result': to_summary(p)})
graph = StageGraph([encode, summarize])
out, trace = graph.run('the quick brown fox jumps')
for e in trace:
    print(f"{e['stage']:10} -> {e['output']}")

## Every stage carries a cost

Each `Stage` has a `service_time`. The graph can project itself onto stage specs for the performance simulator you use in notebook 08 — this is the bridge from *structure* to *performance*.

In [ ]:
from omni_tutorial import build_voice_pipeline
specs = build_voice_pipeline().stage_specs()
for s in specs:
    print(f'{s.name:8} service_time={s.service_time}  replicas={s.replicas}')

## Exercise

Add a third stage `translate` after `encode` that reverses the token list, and run the graph. How many connector transfers do you expect for 3 stages?

In [ ]:
# Solution: N stages -> N-1 edges -> N-1 transfers.
from omni_tutorial import Stage, StageGraph
translate = Stage('translate', 'AR', lambda p: {'tokens': list(reversed(p['tokens']))})
g = StageGraph([encode, translate, summarize])
out, trace = g.run('the quick brown fox jumps')
print('stages:', [e['stage'] for e in trace])
print('transfers:', g.connector.transfers)  # expect 2

## Source lab

Find the stage-graph / deployment abstraction in `vllm_omni/` (look under `engine/` and `deploy/`). Identify what a node and an edge map to in code, and where a transfer function is registered.